# 02 – Task 1: Regression on Community Measures

**Question:** Can we predict continuous community-related node properties
using only features that do NOT require knowing the community structure?

## What we predict (13 targets)
| Target | Meaning |
|--------|---------|
| `cada` | Community-aware degree assortativity |
| `edge_contr` | Edge contribution to community |
| `in_mod_deg` | Intra-community degree |
| `participation` | Participation coefficient |
| `beta_star` | Beta-star community overlap measure |
| `l1`, `l2` | L1/L2 distances from community centroid |
| `kl`, `kl2` | KL divergence measures |
| `hd`, `hd2` | Hellinger distance measures |
| `l12`, `l22` | Squared L1/L2 variants |

## Feature sets used
- **Community-free** (8): `lcc`, `bc`, `cc`, `dc`, `ndc`, `ec`, `eccen`, `core`
- **+ node2vec** (16-dim embedding)
- **+ struc2vec** (16-dim embedding)

## Models
Linear Regression, Ridge, Random Forest, XGBoost, LightGBM

## Metrics
R², Spearman ρ, Kendall τ

In [1]:
# ── CHANGE THIS to switch dataset ──────────────────────────────────────────
DATASET = "facebook"
# ───────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.stats import spearmanr, kendalltau
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

ROOT = Path("..")
PATH_COMBINED = ROOT / f"data/04_models/{DATASET}_combined.parquet"

print(f"Dataset: {DATASET}")

Dataset: facebook


## Step 1 – Load combined dataset

Run `01_data_preparation.ipynb` first to generate the combined parquet file.

In [2]:
df = pd.read_parquet(PATH_COMBINED)

COMMUNITY_FREE   = ["lcc", "bc", "cc", "dc", "ndc", "ec", "eccen", "core"]
REGRESSION_TARGETS = [
    "cada", "edge_contr", "in_mod_deg", "participation",
    "beta_star", "l1", "l2", "kl", "hd", "l12", "l22", "kl2", "hd2",
]
N2V_COLS = [c for c in df.columns if c.startswith("emb_node2vec")]
S2V_COLS = [c for c in df.columns if c.startswith("emb_struc2vec")]

# Feature set: community-free graph features + both embeddings
FEATURES = COMMUNITY_FREE + N2V_COLS + S2V_COLS

print(f"Nodes    : {len(df)}")
print(f"Features : {len(FEATURES)}  "
      f"(8 graph + {len(N2V_COLS)} node2vec + {len(S2V_COLS)} struc2vec)")
print(f"Targets  : {len(REGRESSION_TARGETS)}")

Nodes    : 22470
Features : 40  (8 graph + 16 node2vec + 16 struc2vec)
Targets  : 13


## Step 2 – Train / test split

70% train, 30% test. `random_state=2023` for reproducibility.

In [3]:
X = df[FEATURES + REGRESSION_TARGETS]   # keep targets alongside features in one df

X_train, X_test = train_test_split(X, test_size=0.3, random_state=2023)
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

print(f"Train: {len(X_train)} nodes | Test: {len(X_test)} nodes")

Train: 15729 nodes | Test: 6741 nodes


## Step 3 – Define models

Parameters match those used in the original pipeline (`conf/base/parameters.yml`).

In [4]:
def make_models():
    """Return dict of {name: unfitted model}."""
    return {
        "LinearRegression": LinearRegression(fit_intercept=True, n_jobs=-1),

        "Ridge": Ridge(alpha=1.0, fit_intercept=True, solver="auto", random_state=2023),

        "RandomForest": RandomForestRegressor(
            n_estimators=1000, max_depth=10,
            min_samples_split=2, min_samples_leaf=1,
            n_jobs=-1, random_state=2023
        ),

        "XGBoost": xgb.XGBRegressor(
            n_estimators=1000, max_depth=10,
            learning_rate=0.1, n_jobs=-1,
            random_state=2023, eval_metric="rmse",
            verbosity=0,
        ),

        "LightGBM": lgb.LGBMRegressor(
            boosting_type="gbdt", objective="regression",
            num_leaves=10, n_jobs=-1,
        ),
    }

print("Models defined:", list(make_models().keys()))

Models defined: ['LinearRegression', 'Ridge', 'RandomForest', 'XGBoost', 'LightGBM']


## Step 4 – Train and evaluate

For each **target** × **model** combination we compute:
- **R²** – coefficient of determination (1 = perfect, 0 = baseline mean)
- **Spearman ρ** – rank correlation (robust to outliers)
- **Kendall τ** – another rank correlation, more conservative

In [5]:
records = []

for target in REGRESSION_TARGETS:
    y_train = X_train[target]
    y_test  = X_test[target]
    X_tr    = X_train[FEATURES]
    X_te    = X_test[FEATURES]

    for model_name, model in make_models().items():
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)

        records.append({
            "target"    : target,
            "model"     : model_name,
            "R2"        : round(r2_score(y_test, y_pred), 4),
            "Spearman"  : round(spearmanr(y_test, y_pred).statistic, 4),
            "Kendall"   : round(kendalltau(y_test, y_pred).statistic, 4),
        })

        print(f"  {target:15s} | {model_name:16s} | "
              f"R²={records[-1]['R2']:.3f}  "
              f"ρ={records[-1]['Spearman']:.3f}  "
              f"τ={records[-1]['Kendall']:.3f}")

results = pd.DataFrame(records)
print(f"\nDone. {len(results)} rows in results table.")

  cada            | LinearRegression | R²=0.237  ρ=0.624  τ=0.479
  cada            | Ridge            | R²=0.220  ρ=0.619  τ=0.477
  cada            | RandomForest     | R²=0.408  ρ=0.684  τ=0.542
  cada            | XGBoost          | R²=0.457  ρ=0.701  τ=0.560
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000793 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9644
[LightGBM] [Info] Number of data points in the train set: 15729, number of used features: 40
[LightGBM] [Info] Start training from score 1.149959
  cada            | LightGBM         | R²=0.448  ρ=0.681  τ=0.538
  edge_contr      | LinearRegression | R²=0.307  ρ=0.646  τ=0.496
  edge_contr      | Ridge            | R²=0.299  ρ=0.642  τ=0.494
  edge_contr      | RandomForest     | R²=0.497  ρ=0.713  τ=0.564
  edge_contr      | XGBoost          | R²=0.548  ρ=0.728  τ=0.582
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the ov

## Step 5 – Results table

In [6]:
# Pivot: rows = target, columns = model × metric
pivot = results.pivot_table(
    index="target", columns="model", values=["R2", "Spearman", "Kendall"]
)
pivot.columns = [f"{metric}_{model}" for metric, model in pivot.columns]
pivot = pivot.round(4)
pivot

,Kendall_LightGBM,Kendall_LinearRegression,Kendall_RandomForest,Kendall_Ridge,Kendall_XGBoost,R2_LightGBM,R2_LinearRegression,R2_RandomForest,R2_Ridge,R2_XGBoost,Spearman_LightGBM,Spearman_LinearRegression,Spearman_RandomForest,Spearman_Ridge,Spearman_XGBoost
target,,,,,,,,,,,,,,,
beta_star,0.5645,0.5236,0.5780,0.5250,0.6245,0.5662,0.3849,0.5532,0.3772,0.5947,0.7613,0.7249,0.7708,0.7255,0.8071
cada,0.5377,0.4792,0.5422,0.4770,0.5600,0.4480,0.2372,0.4085,0.2201,0.4567,0.6807,0.6240,0.6837,0.6188,0.7005
edge_contr,0.5625,0.4959,0.5644,0.4939,0.5816,0.5174,0.3068,0.4969,0.2992,0.5485,0.7119,0.6459,0.7133,0.6417,0.7277
hd,0.6723,0.5647,0.7022,0.5636,0.7580,0.7378,0.5842,0.7597,0.5743,0.8246,0.8506,0.7511,0.8684,0.7499,0.9084
hd2,0.7740,0.7185,0.7822,0.7175,0.8277,0.8777,0.7481,0.8823,0.7289,0.9211,0.9298,0.8943,0.9328,0.8939,0.9561
in_mod_deg,0.7802,0.6751,0.7921,0.6691,0.8514,0.8980,0.7640,0.8922,0.7189,0.9190,0.9316,0.8598,0.9354,0.8585,0.9623
kl,0.6625,0.5331,0.7062,0.5332,0.7831,0.7088,0.4747,0.7673,0.4739,0.8640,0.8341,0.7149,0.8655,0.7148,0.9206
kl2,0.7121,0.6352,0.7313,0.6320,0.8049,0.7869,0.6318,0.8137,0.6277,0.8974,0.8836,0.8227,0.8924,0.8202,0.9392
l1,0.6342,0.5632,0.6502,0.5605,0.7107,0.7180,0.5708,0.7187,0.5560,0.7728,0.8225,0.7534,0.8308,0.7507,0.8756


In [7]:
# Best model per target (by Spearman)
best = (
    results.sort_values("Spearman", ascending=False)
           .groupby("target")
           .first()[["model", "R2", "Spearman", "Kendall"]]
)
print("Best model per target (by Spearman ρ):")
best

Best model per target (by Spearman ρ):


,model,R2,Spearman,Kendall
target,,,,
beta_star,XGBoost,0.5947,0.8071,0.6245
cada,XGBoost,0.4567,0.7005,0.5600
edge_contr,XGBoost,0.5485,0.7277,0.5816
hd,XGBoost,0.8246,0.9084,0.7580
hd2,XGBoost,0.9211,0.9561,0.8277
in_mod_deg,XGBoost,0.9190,0.9623,0.8514
kl,XGBoost,0.8640,0.9206,0.7831
kl2,XGBoost,0.8974,0.9392,0.8049
l1,XGBoost,0.7728,0.8756,0.7107


In [8]:
# Save results
PATH_OUT = ROOT / f"data/05_reporting/{DATASET}_task1.parquet"
results.to_parquet(PATH_OUT, index=False)
print(f"Saved to {PATH_OUT}")

Saved to ../data/05_reporting/facebook_task1.parquet
